# Vision Transformers

#Steps:
Patch extraction

1.   Patch embedding
2. CLS token
3. Positional embeddings
4. Single-head attention first
5. Multi-head attention
6. MLP block
7. One Transformer encoder block
8. Stack 12 encoder blocks
9. Classification head
10. Training loop
11. Compare with 97% CNN


##1. patch extarction

*   load the image
*   make the patches for every image
the size of the image patches like that
[channels,patch rows,patch rows, patch columns, patch height , patch width]
*   flatten the patches so they are like the transformers tokens







#2. CLS token
CLS token as a learned summary representation of the whole image
it has the same size as the other tokens the features that will be in it will the most important ones that we get from the other tokens so . it contains information about all the other tokens

##3. positional embedding:
in here the positional embeddings are the position informatin that the transformer wants to know the position of each token the original transformer they provided as give not learnable but in vit they thought that lets make it a learnable as maybe the model think this representation are not good for the model

## 4. Single-head attention
the idea behind it is that for all the patches i have to make their q ,k and value vector but at this point we need to just make it as learnable parameter

define a class for vision transformer

In [2]:
!unzip "balanced_dataset.zip" -d dataset

Streaming output truncated to the last 5000 lines.
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (348).JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (349).JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (349)_Shear.JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (35).JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (350).JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (351).JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (351)_ElasticDistortion.JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (352).JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (353).JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (353)_Flip.JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (354).JPG  
  inflating: dataset/balanced_dataset/data/Grape_Black_rot/image (354)_Ske

In [24]:
from torch import nn
class EncoderBlock(nn.Module):
  def __init__(self,embed_dim=512,head_nums= 8):
    super().__init__()
    self.WQ = nn.Linear(self.embed_dim,self.embed_dim)
    self.WK = nn.Linear(self.embed_dim,self.embed_dim)
    self.WV = nn.Linear(self.embed_dim,self.embed_dim)
    self.WO = nn.Linear(self.embed_dim,self.embed_dim)
    self.layer_norm1 = nn.LayerNorm(self.embed_dim)
    self.layer_norm2 = nn.LayerNorm(self.embed_dim)
    self.mlp = nn.Sequential(
        nn.Linear(self.embed_dim,self.embed_dim*4),
        nn.GELU(),
        nn.Linear(self.embed_dim*4,self.embed_dim))

  def foreward(self,x):
    # layer norm
    residual = patches
    patches = self.layer_norm1(patches)
    # multi head attention
    Q = self.WQ(patches)
    K = self.WK(patches)
    V = self.WV(patches)
    # Multi-head attention splits the Q, K, and V feature dimensions into multiple heads
    # so that the model can learn multiple attention relationships between the patches in parallel.
    Q =Q.reshape(self.B,self.patch_nums+1,self.head_nums,-1)
    Q =Q.permute(0,2,1,3)
    K =K.reshape(self.B,self.patch_nums+1,self.head_nums,-1)
    K =K.permute(0,2,1,3)
    V =V.reshape(self.B,self.patch_nums+1,self.head_nums,-1)
    V =V.permute(0,2,1,3)
    scores = torch.matmul(Q,K.transpose(-2,-1))
    # the next step for scaling the score so we are avoinding the gradient problems
    attention = scores/math.sqrt(self.embed_dim//self.head_nums)
    attention = torch.softmax(attention,dim=-1)
    output = torch.matmul(attention,V)
    output = output.permute(0,2,1,3)

    # collecting the features again together
    output = output.reshape(self.B,self.patch_nums+1,-1)
    output = self.WO(output)

    # residual connection
    patches =  residual + output
    residual = patches
    # layer norm
    patches = self.layer_norm2(patches)
    # mlp
    patches = self.mlp(patches)
    # residual connection
    patches = residual + patches
    return patches

In [25]:
from torch import nn
import math
class VisionTransformer(nn.Module):
  def __init__(self,patch_size = 16,channels = 3,embed_dim = 512, image_size=224):
    super().__init__()
    self.patch_size = patch_size
    self.patch_dim = channels * patch_size * patch_size
    self.patch_nums= (image_size//patch_size)**2
    self.embed_dim = embed_dim

    self.patch_embeddings= nn.Linear(self.patch_dim , self.embed_dim )
    self.cls = nn.Parameter(torch.randn(1,1,self.embed_dim))
    self.POS = nn.Parameter(torch.randn(1,self.patch_nums+1,self.embed_dim))
    self.classifier = nn.Linear(self.embed_dim,8)
    self.encoders = nn.ModuleList([
    EncoderBlock()
    for _ in range(6)
])

  def make_patches(self,images):
    """ take the image and patch it pased on the patch size and return the flatten patches """
    B, C, H, W = images.shape
    patches = images.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
    patches=patches.permute(0,2,3,1,4,5)
    patches = patches.reshape(B, self.patch_nums,-1)
    return patches

  def forward(self,x):
    B,C,H,W = x.shape
    self.B = B
    patches = self.make_patches(x)
    patches_embbedding = self.patch_embeddings(patches)
    cls = self.cls.expand(self.B,-1,-1)
    all_patches = torch.cat((cls,patches_embbedding),dim=1)
    pos=self.POS.expand(self.B,-1,-1)
    all_patches = all_patches + pos

    # six encoder blocks
    for encoder in self.encoders:
      all_patches = encoder(all_patches)
    cls_token = all_patches[:,0]
    logits = self.classifier(cls_token)
    return logits


In [26]:
from PIL import Image
from torchvision.datasets import ImageFolder
from torchvision import transforms, torch
from torch.utils.data import random_split
from torch.utils.data import DataLoader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
dataset = ImageFolder(root="dataset/balanced_dataset/data/", transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])



In [27]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [28]:
images,labels = next(iter(train_loader))
print(images.shape)

torch.Size([32, 3, 224, 224])


In [29]:
import torch

print(torch.cuda.is_available())
print(next(model.parameters()).device)

True
cuda:0


In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 20
model = VisionTransformer(patch_size=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss()

AttributeError: 'EncoderBlock' object has no attribute 'embed_dim'

In [8]:

for epoch in range (epochs):
  model.train()
  train_correct = 0
  train_total = 0
  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    pred = model(images)
    loss = loss_fn(pred,labels)
    predicted_classes = torch.argmax(pred, dim=1)
    train_correct += (predicted_classes == labels).sum().item()
    train_total += labels.size(0)
    train_accuracy = train_correct / train_total
    loss.backward()
    optimizer.step()
# validation
  model.eval()
  val_correct = 0
  val_total = 0
  with torch.no_grad():
      for images, labels in test_loader:
          images = images.to(device)
          labels = labels.to(device)
          outputs = model(images)
          predictions = outputs.argmax(dim=1)
          val_correct += (predictions == labels).sum().item()
          val_total += labels.size(0)
  val_accuracy = val_correct / val_total
  print(
        f"Epoch {epoch+1}/{ epochs} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

Epoch 1/20 | Train Acc: 0.3416 | Val Acc: 0.5306
Epoch 2/20 | Train Acc: 0.6356 | Val Acc: 0.6749
Epoch 3/20 | Train Acc: 0.7399 | Val Acc: 0.7225
Epoch 4/20 | Train Acc: 0.7824 | Val Acc: 0.7514
Epoch 5/20 | Train Acc: 0.8032 | Val Acc: 0.7610
Epoch 6/20 | Train Acc: 0.8011 | Val Acc: 0.7676
Epoch 7/20 | Train Acc: 0.8217 | Val Acc: 0.8230
Epoch 8/20 | Train Acc: 0.6007 | Val Acc: 0.1071
Epoch 9/20 | Train Acc: 0.1775 | Val Acc: 0.2047
Epoch 10/20 | Train Acc: 0.2035 | Val Acc: 0.2589
Epoch 11/20 | Train Acc: 0.2252 | Val Acc: 0.2204
Epoch 12/20 | Train Acc: 0.2447 | Val Acc: 0.2270
Epoch 13/20 | Train Acc: 0.2162 | Val Acc: 0.1274
Epoch 14/20 | Train Acc: 0.2162 | Val Acc: 0.2486
Epoch 15/20 | Train Acc: 0.2321 | Val Acc: 0.2080
Epoch 16/20 | Train Acc: 0.2656 | Val Acc: 0.2382
Epoch 17/20 | Train Acc: 0.2249 | Val Acc: 0.1762
Epoch 18/20 | Train Acc: 0.2552 | Val Acc: 0.2758


KeyboardInterrupt: 